# CSIT 359/553 Homework 4: Plotting with folium


In [ ]:
import numpy as np
import pandas as pd
pd.options.display.max_rows = 10

import matplotlib.pyplot as plt
import seaborn as sns


import folium

## About the Data Set: Chicago Food Inspections
Link: https://data.cityofchicago.org/Health-Human-Services/Food-Inspections/4ijn-s7e5

This information is derived from inspections of restaurants and other food establishments in Chicago from January 1, 2010 to the present. Inspections are performed by staff from the Chicago Department of Public Health’s Food Protection Program using a standardized procedure. The results of the inspection are inputted into a database, then reviewed and approved by a State of Illinois Licensed Environmental Health Practitioner (LEHP). For descriptions of the data elements included in this set, go to http://bit.ly/tS9IE8

Note about 7/1/2018 change to food inspection procedures that affects the data in this dataset: http://bit.ly/2yWd2JB

Disclaimer: Attempts have been made to minimize any and all duplicate inspection reports. However, the dataset may still contain such duplicates and the appropriate precautions should be exercised when viewing or analyzing these data. The result of the inspections (pass, pass with conditions or fail) as well as the violations noted are based on the findings identified and reported by the inspector at the time of the inspection, and may not reflect the findings noted at other times. For more information about Food Inspections, go to https://www.cityofchicago.org/city/en/depts/cdph/provdrs/healthy_restaurants/svcs/food-protection-services.html.

Columns in this Dataset

| Column Name     | Description       | Type       |
|-----------------|-------------------|------------|
| Inspection ID   |                   | Number     |
| DBA Name        | Doing Business As | Plain Text |
| AKA Name        | Also Known As     | Plain Text |
| License #       |                   | Number     |
| Facility Type   |                   | Plain Text |
| Risk            |                   | Plain Text |
| Address         |                   | Plain Text |
| City            |                   | Plain Text |
| State           |                   | Plain Text |
| Zip             |                   | Number     |
| Inspection Date |                   | Date&Time  |
| Results         |                   | Plain Text |
| Violations      |                   | Plain Text |
| Latitude        |                   | Number     |
| Longitude       |                   | Number     |
| Location        |                   | Location   |

### Load the data

In [ ]:
df = pd.read_csv(
    'Food_Inspections.csv',
    engine='python',
    on_bad_lines='skip'
)

In [ ]:
df.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
0,2500341,OAZAS,OAZAS,2476381.0,Restaurant,Risk 1 (High),3057 W LAWRENCE AVE,CHICAGO,IL,60625.0,04/16/2021,Non-Inspection,No Entry,NaN,41.968367,-87.705891,"(-87.70589093652268, 41.96836672599028)"
1,2484567,ROYALTY,ROYALTY,1306130.0,Restaurant,Risk 1 (High),3810 W 63RD ST,CHICAGO,IL,60629.0,02/05/2021,Non-Inspection,No Entry,NaN,41.778837,-87.718361,"(-87.71836138998039, 41.778836516734856)"
2,2473041,ROSATI'S GRANT PARK,ROSATI'S,2762683.0,Restaurant,Risk 1 (High),23 E ADAMS ST,CHICAGO,IL,60603.0,01/22/2021,License,Pass,NaN,41.879391,-87.626848,"(-87.62684825563626, 41.879391313239694)"
3,2463977,HWA WON,HWA WON,2341742.0,Restaurant,Risk 1 (High),2519 W PETERSON AVE,CHICAGO,IL,60659.0,12/21/2020,Non-Inspection,No Entry,NaN,41.990368,-87.692981,"(-87.6929808527407, 41.99036795715765)"
4,2463877,NO.7,NO.7,2560610.0,Restaurant,Risk 1 (High),2485 N CLARK ST,CHICAGO,IL,60614.0,12/16/2020,Canvass,Out of Business,NaN,41.927834,-87.641659,"(-87.6416588091054, 41.92783381766517)"


In [ ]:
df.Results.unique()

array(['No Entry', 'Pass', 'Out of Business', 'Not Ready', 'Fail',
       'Pass w/ Conditions', 'Business Not Located'], dtype=object)

### Extract the columns of inspection date and inspection results

In [ ]:
df['Inspection Date'] = pd.to_datetime(df['Inspection Date'])

In [ ]:
df.set_index('Inspection Date',inplace=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 137455 entries, 2021-04-16 to 2014-10-22
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Inspection ID    137455 non-null  int64  
 1   DBA Name         137455 non-null  object 
 2   AKA Name         136531 non-null  object 
 3   License #        137449 non-null  float64
 4   Facility Type    136019 non-null  object 
 5   Risk             137426 non-null  object 
 6   Address          137455 non-null  object 
 7   City             137326 non-null  object 
 8   State            137404 non-null  object 
 9   Zip              137429 non-null  float64
 10  Inspection Type  137454 non-null  object 
 11  Results          137455 non-null  object 
 12  Violations       100761 non-null  object 
 13  Latitude         136973 non-null  float64
 14  Longitude        136973 non-null  float64
 15  Location         136973 non-null  object 
dtypes: float64(4), int64(1

## Create a map of Chicago by with the following setting:

1. Set the centre of your map to be [41.8781, -87.6298], which is the center of Chicago.
2. Initialize the zoom level to 10.

In [ ]:
import folium

# Create map centered on Chicago
chicago_map = folium.Map(
    location=[41.8781, -87.6298],
    zoom_start=10
)

# Display map
chicago_map

##Find all the good restaurants inspected in **April 2021** and mark them in the map of Chicago.



In [ ]:
from folium.plugins import MarkerCluster

good_restaurants = df[
    (df.index.year == 2021) &
    (df.index.month == 4) &
    (df['Results'] == 'Pass') &
    (df['Facility Type'].str.contains('Restaurant', case=False, na=False)) &
    (df['Latitude'].notna()) &
    (df['Longitude'].notna())
]

chicago_map = folium.Map(location=[41.8781, -87.6298], zoom_start=12)

marker_cluster = MarkerCluster().add_to(chicago_map)

for _, row in good_restaurants.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=row['DBA Name']
    ).add_to(marker_cluster)

chicago_map

##Create a choropleth map to indicate the number of good restaurants in each area of Chicago.



In [ ]:
import json

# Convert ZIP to string (remove .0)
good_restaurants['Zip'] = good_restaurants['Zip'].astype(int).astype(str)

# Count good restaurants by ZIP
zip_counts = good_restaurants.groupby('Zip').size().reset_index(name='Count')

# Load GeoJSON file
with open('chicago.json', 'r') as f:
    chicago_geo = json.load(f)

# Create base map
chicago_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

# Create choropleth map
folium.Choropleth(
    geo_data=chicago_geo,
    data=zip_counts,
    columns=['Zip', 'Count'],
    key_on='feature.properties.ZIP',
    fill_color='YlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Number of Good Restaurants'
).add_to(chicago_map)

# Display map
chicago_map

/tmp/ipykernel_4958/3522590467.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  good_restaurants['Zip'] = good_restaurants['Zip'].astype(int).astype(str)
